# Kidney Disease Risk Prediction
### A clean Google Colab notebook to:

- install requirements
- Load the dataset
- Inspect head and tail
- Check for missing values
- train,test and export model
- Understand schema design ideas - Random Forest Model

**Author:** Group 12  
**Date:** 2nd November 2025  
**Course:** Pipeline

## Project Overview

This notebook implements a Random Forest classifier for kidney disease prediction as part of the database assignment. The model achieves excellent performance and will be integrated with a FastAPI backend for real-time predictions.

### Dataset Features
- **Demographics**: Age
- **Lab Results**: Creatinine Level, BUN, GFR, Urine Output  
- **Medical History**: Diabetes, Hypertension
- **Target**: CKD Status prediction

In [4]:
# Install required packages
%pip install kagglehub pandas numpy scikit-learn joblib matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [5]:
# Dataset Download and Setup
import kagglehub
import os
import pandas as pd
import numpy as np

# Download latest version of kidney disease risk dataset
path = kagglehub.dataset_download("miadul/kidney-disease-risk-dataset")

print("Path to dataset files:", path)

# Load the dataset
data_path = os.path.join(path, 'kidney_disease_dataset.csv')
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst few rows:")
print(df.head())
print("\nDataset info:")
print(df.info())

C:\Users\evotech\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\evotech\.cache\kagglehub\datasets\miadul\kidney-disease-risk-dataset\versions\1
Dataset shape: (2304, 9)

Column names: ['Age', 'Creatinine_Level', 'BUN', 'Diabetes', 'Hypertension', 'GFR', 'Urine_Output', 'CKD_Status', 'Dialysis_Needed']

First few rows:
   Age  Creatinine_Level   BUN  Diabetes  Hypertension   GFR  Urine_Output  \
0   71              0.30  40.9         0             1  46.8        1622.0   
1   34              1.79  17.1         0             0  43.8        1428.0   
2   80              2.67  15.0         0             1  78.2        1015.0   
3   40              0.97  31.1         0             1  92.8        1276.0   
4   43              2.05  22.8         1             1  62.2        1154.0   

   CKD_Status  Dialysis_Needed  
0           1                0  
1           1                0  
2           1                0  
3           1                0  
4           0                0  

Dataset info:
<class 'pandas.core.frame.Data

In [6]:
# Data Analysis for Database Design
print("=== DATASET ANALYSIS FOR DATABASE DESIGN ===")
print(f"Total Records: {len(df)}")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicate Rows: {df.duplicated().sum()}")

# Feature categorization for database schema
continuous_features = ['Age', 'Creatinine_Level', 'BUN', 'GFR', 'Urine_Output']
categorical_features = ['Diabetes', 'Hypertension']
target_features = ['CKD_Status', 'Dialysis_Needed']

print(f"\nContinuous Features: {continuous_features}")
print(f"Categorical Features: {categorical_features}")
print(f"Target Features: {target_features}")

# Target distribution
print("\n=== TARGET VARIABLE DISTRIBUTION ===")
print("CKD_Status Distribution:")
print(df['CKD_Status'].value_counts())
print("\nDialysis_Needed Distribution:")  
print(df['Dialysis_Needed'].value_counts())

=== DATASET ANALYSIS FOR DATABASE DESIGN ===
Total Records: 2304
Missing Values: 0
Duplicate Rows: 0

Continuous Features: ['Age', 'Creatinine_Level', 'BUN', 'GFR', 'Urine_Output']
Categorical Features: ['Diabetes', 'Hypertension']
Target Features: ['CKD_Status', 'Dialysis_Needed']

=== TARGET VARIABLE DISTRIBUTION ===
CKD_Status Distribution:
CKD_Status
1    1172
0    1132
Name: count, dtype: int64

Dialysis_Needed Distribution:
Dialysis_Needed
0    2273
1      31
Name: count, dtype: int64


## Data Preprocessing for Machine Learning

Before training the Random Forest model, we need to prepare the data by handling missing values, encoding categorical variables, and splitting the dataset.

In [7]:
# Data Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# Check for missing values and handle them
print("Missing values per column:")
print(df.isnull().sum())

# Handle missing values if any (fill with median for numeric, mode for categorical)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)

print("\nAfter handling missing values:")
print(df.isnull().sum().sum())

Missing values per column:
Age                 0
Creatinine_Level    0
BUN                 0
Diabetes            0
Hypertension        0
GFR                 0
Urine_Output        0
CKD_Status          0
Dialysis_Needed     0
dtype: int64

After handling missing values:
0


In [8]:
# Feature Engineering and Data Splitting
# Define features and target
feature_columns = ['Age', 'Creatinine_Level', 'BUN', 'GFR', 'Urine_Output', 'Diabetes', 'Hypertension']
target_column = 'CKD_Status'  # We'll focus on CKD prediction for this assignment

# Prepare features and target
X = df[feature_columns]
y = df[target_column]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("\nFeature columns:", feature_columns)
print("Target column:", target_column)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Class distribution in training set:")
print(y_train.value_counts())

Feature matrix shape: (2304, 7)
Target vector shape: (2304,)

Feature columns: ['Age', 'Creatinine_Level', 'BUN', 'GFR', 'Urine_Output', 'Diabetes', 'Hypertension']
Target column: CKD_Status

Training set size: 1843
Test set size: 461
Class distribution in training set:
CKD_Status
1    937
0    906
Name: count, dtype: int64


## Random Forest Model Training

We'll train and optimize a Random Forest classifier for kidney disease prediction.

In [9]:
# Random Forest Model Training
print("=== TRAINING RANDOM FOREST MODEL ===\n")

# Initialize Random Forest with optimal parameters
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

# Train the model
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\nRandom Forest model trained successfully!")

=== TRAINING RANDOM FOREST MODEL ===

Model Accuracy: 1.0000 (100.00%)

Random Forest model trained successfully!


In [ ]:
# Model Evaluation
print("=== MODEL EVALUATION ===")

# Detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Feature importance analysis
print("\nFeature Importance (Ranked):")
feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

for idx, row in feature_importance.iterrows():
    print(f"{row['Feature']}: {row['Importance']:.4f}")

# Summary statistics
print(f"\n=== MODEL SUMMARY ===")
print(f"✅ Model evaluation completed!")
print(f"🎯 Final Accuracy: {accuracy*100:.2f}%")
print(f"📊 Total Test Samples: {len(y_test)}")
print(f"🔍 Most Important Feature: {feature_importance.iloc[0]['Feature']}")

=== MODEL EVALUATION ===
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       226
           1       1.00      1.00      1.00       235

    accuracy                           1.00       461
   macro avg       1.00      1.00      1.00       461
weighted avg       1.00      1.00      1.00       461


Confusion Matrix:
[[226   0]
 [  0 235]]

Feature Importance:
GFR: 0.6017
BUN: 0.2352
Creatinine_Level: 0.1291
Urine_Output: 0.0188
Age: 0.0122
Hypertension: 0.0015
Diabetes: 0.0015

 Model evaluation completed!
 Final Accuracy: 100.00%


In [11]:
# Save Model for API Deployment
print("SAVING MODEL FOR API DEPLOYMENT\n")

# Create models directory
os.makedirs("models", exist_ok=True)

# Save the trained model
model_path = "models/kidney_disease_rf_model.joblib"
joblib.dump(rf_model, model_path)
print(f"✓ Model saved: {model_path}")

# Save feature names
feature_names_path = "models/feature_names.joblib"
joblib.dump(feature_columns, feature_names_path)
print(f"✓ Feature names saved: {feature_names_path}")

# Save model metadata
model_metadata = {
    "model_type": "RandomForestClassifier",
    "accuracy": float(accuracy),
    "feature_columns": feature_columns,
    "target_column": target_column,
    "training_samples": int(X_train.shape[0]),
    "test_samples": int(X_test.shape[0]),
    "model_params": {
        "n_estimators": 100,
        "max_depth": 10,
        "random_state": 42
    },
    "training_date": pd.Timestamp.now().isoformat()
}

metadata_path = "models/model_metadata.joblib"
joblib.dump(model_metadata, metadata_path)
print(f"✓ Metadata saved: {metadata_path}")

print(f"\n Model ready for deployment!")
print(f"Final Accuracy: {accuracy*100:.2f}%")
print(f" Features: {feature_columns}")
print(f" Predicts: {target_column}")

SAVING MODEL FOR API DEPLOYMENT

✓ Model saved: models/kidney_disease_rf_model.joblib
✓ Feature names saved: models/feature_names.joblib
✓ Metadata saved: models/model_metadata.joblib

 Model ready for deployment!
Final Accuracy: 100.00%
 Features: ['Age', 'Creatinine_Level', 'BUN', 'GFR', 'Urine_Output', 'Diabetes', 'Hypertension']
 Predicts: CKD_Status


In [12]:
# Test Model Loading and Prediction
print("=== TESTING MODEL LOADING ===\n")

# Load the saved model
loaded_model = joblib.load("models/kidney_disease_rf_model.joblib")
loaded_features = joblib.load("models/feature_names.joblib")
loaded_metadata = joblib.load("models/model_metadata.joblib")

print("✓ Model loaded successfully!")
print(f"✓ Features: {loaded_features}")
print(f"✓ Model accuracy: {loaded_metadata['accuracy']*100:.2f}%")

# Test prediction with sample data
sample_data = {
    "Age": 65,
    "Creatinine_Level": 2.1,
    "BUN": 45.2,
    "GFR": 35.8,
    "Urine_Output": 800.0,
    "Diabetes": 1,
    "Hypertension": 1
}

sample_df = pd.DataFrame([sample_data])
prediction = loaded_model.predict(sample_df)[0]
probability = loaded_model.predict_proba(sample_df)[0]

print(f"\nSample Prediction Test:")
print(f"Input: {sample_data}")
print(f"Prediction: {'CKD' if prediction == 1 else 'No CKD'}")
print(f"Probability: {probability[1]:.4f}")
print("\n Model is ready for API integration!")

=== TESTING MODEL LOADING ===

✓ Model loaded successfully!
✓ Features: ['Age', 'Creatinine_Level', 'BUN', 'GFR', 'Urine_Output', 'Diabetes', 'Hypertension']
✓ Model accuracy: 100.00%

Sample Prediction Test:
Input: {'Age': 65, 'Creatinine_Level': 2.1, 'BUN': 45.2, 'GFR': 35.8, 'Urine_Output': 800.0, 'Diabetes': 1, 'Hypertension': 1}
Prediction: CKD
Probability: 1.0000

 Model is ready for API integration!
